
# LLM-Based KPI Extraction from OCR JSON (Vertex AI Gemini)

This notebook:

1. Loads a local **Excel file** (`Pre Disbursement Checklist.xlsx`) with at least:
   - Column 2: `document_type` (name doesn't matter, first column is treated as this)
   - Column 3: `kpi_description` (textual description of the KPI / information to extract)
2. Collects **all `kpi_description` values** into a single list.
3. Reads OCR result JSONs from your **GCS bucket** (`vision-ocr-results/`).
4. For **each document** inside each OCR JSON, calls **Gemini 1.5 Flash on Vertex AI** with:
   - the document type label from the JSON (e.g., `APPLICATION FORM_46`)
   - the full OCR text for that document
   - the full list of KPI descriptions from the Excel file
5. The LLM is instructed to:
   - Identify **which KPIs are relevant** to that document type and text
   - Extract a **value** for each relevant KPI (best guess)
6. Writes a local **CSV** with rows:
   - `applicant_id`
   - `document_type` (from JSON)
   - `kpi_description` (from Excel, for KPIs that were detected)
   - `field_name` (LLM's human-readable label, if different)
   - `value` (or `NA` if not found)


In [ ]:

import os
import json
from pathlib import Path
from typing import Dict, List, Any

import pandas as pd
from google.cloud import storage
import vertexai
from vertexai.generative_models import GenerativeModel

# -----------------------------
# AUTHENTICATION
# -----------------------------
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "../../turing-agent-358210-a38a4820a9ce.json"

PROJECT_ID = "turing-agent-358210"
LOCATION = "us-central1"
GEMINI_MODEL_NAME = "gemini-2.5-flash"

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

# -----------------------------
# CONFIG
# -----------------------------
BUCKET_NAME = "capstone-ii-applicant-documents"
RESULTS_PREFIX = "vision-ocr-results/"

EXCEL_CONFIG_PATH = Path("../../Pre Disbursement Checklist.xlsx")
OUTPUT_CSV_PATH = Path("../data/gemini_llm_kpi_anymatch_extracted.csv")


model = GenerativeModel(GEMINI_MODEL_NAME)

print("Vertex AI + Gemini model initialized.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Model:", GEMINI_MODEL_NAME)

Vertex AI + Gemini model initialized.


c:\Users\luisd\anaconda3\envs\capstone_env\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [7]:

# -----------------------------
# LOAD EXCEL KPI CONFIG
# -----------------------------
if not EXCEL_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Excel config not found at: {EXCEL_CONFIG_PATH.resolve()}")

config_df = pd.read_excel(EXCEL_CONFIG_PATH, sheet_name="Pre-Disbursement")

if config_df.shape[1] < 2:
    raise ValueError(
        "Excel file must have at least 2 columns: "
        "first = document_type, second = kpi_description."
    )

doc_col, kpi_col = config_df.columns[:2]
config_df = config_df.rename(columns={doc_col: "document_type", kpi_col: "kpi_description"})

config_df["document_type"] = config_df["document_type"].astype(str).str.strip()
config_df["kpi_description"] = config_df["kpi_description"].astype(str).str.strip()

print("Sample of KPI config:")
display(config_df.head())

all_kpis: List[str] = (
    config_df["kpi_description"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

print(f"Total unique KPI descriptions: {len(all_kpis)}")
print("First 10 KPI descriptions:")
for k in all_kpis[:10]:
    print(" -", k)


Sample of KPI config:


,document_type,kpi_description,KPI & data sources to cross verify (With Client Remarks)
0,1,Disbursement memo,Borrower name- application form-
1,1,Disbursement memo,repayment mode-nach form and PDC-
2,1,Disbursement memo,Roi-Loan proposal note(LPN)-
3,1,Disbursement memo,fees details-to be checked with core system-Pr...
4,1,Disbursement memo,beneficiary details-check from legal appraisal...


Total unique KPI descriptions: 28
First 10 KPI descriptions:
 - Disbursement memo
 - Builder demand and permission to mortgage
 - Deviation Approval Mails
 - MOTD challan and Draft copy - vetted
 - External FI reports /RCU Report
 - NACH and PDC
 - FBD/EMI commencement  letter
 - Disbursement request form/Mail Confirmation from Customer
 - External technical valuation report
 - Loan agreement schedule duly signed.


In [8]:

# -----------------------------
# GCS HELPERS
# -----------------------------
def get_gcs_client():
    return storage.Client()

def list_ocr_json_blobs(bucket, prefix: str):
    return [
        blob for blob in bucket.list_blobs(prefix=prefix)
        if blob.name.lower().endswith(".json")
    ]

def infer_applicant_id(blob_name: str) -> str:
    parts = blob_name.split("/")
    for p in parts:
        if p.lower().startswith("applicant"):
            return p
    return parts[-2] if len(parts) >= 2 else blob_name

def load_ocr_json(blob) -> Dict[str, Any]:
    return json.loads(blob.download_as_text())


In [9]:

# -----------------------------
# GEMINI HELPER (NEW APPROACH)
# -----------------------------
def call_gemini_for_document(
    document_label: str,
    ocr_text: str,
    kpi_descriptions: List[str],
    max_ocr_chars: int = 12000,
    max_kpis: int = 200,
) -> List[Dict[str, Any]]:
    if not ocr_text:
        ocr_text = ""

    ocr_text_input = ocr_text[:max_ocr_chars]

    if len(kpi_descriptions) > max_kpis:
        kpis_input = kpi_descriptions[:max_kpis]
    else:
        kpis_input = kpi_descriptions

    payload = {
        "document_label": document_label,
        "kpi_descriptions": kpis_input,
        "ocr_text": ocr_text_input,
    }

    system_msg = """
You are an assistant for a loan document validation system.

You are given:
- A document label (e.g., "Disbursement memo", "Loan Application Form", "APPLICATION FORM_46").
- OCR text for that document.
- A list of KPI descriptions that describe checks or pieces of information
  that might appear in *some* document types.

Your task for THIS SINGLE DOCUMENT is:
1. Decide which KPI descriptions are actually relevant to this document,
   based on the document label and the OCR text.
2. For each relevant KPI description:
   - Return the KPI description verbatim.
   - Infer a concise field name that represents the main value to extract.
   - Extract the best-guess value from the OCR text.
3. If a KPI clearly does NOT apply to this document, do not include it
   in the output.
4. If a KPI seems relevant but the value isn't clearly present, include it
   with value = null.

Return STRICT JSON with this exact structure:
{
  "document_label": "string",
  "kpis": [
    {
      "kpi_description": "string",
      "field_name": "string",
      "value": "string or null"
    },
    ...
  ]
}

Do not add any other top-level keys. Do not include explanations.
""".strip()

    prompt = (
        system_msg
        + "\n\n--- INPUT JSON ---\n"
        + json.dumps(payload, ensure_ascii=False)
        + "\n------------------\n"
        "Respond ONLY with the JSON object in the required format."
    )

    response = model.generate_content(prompt)
    raw = (response.text or "").strip()

    try:
        parsed = json.loads(raw)
    except Exception:
        start = raw.find("{")
        end = raw.rfind("}") + 1
        parsed = json.loads(raw[start:end])

    kpi_entries = parsed.get("kpis", []) or []

    records: List[Dict[str, Any]] = []
    for item in kpi_entries:
        kpi_desc = str(item.get("kpi_description", "")).strip()
        field_name = str(item.get("field_name", "")).strip()
        value = item.get("value", None)

        if not kpi_desc:
            continue

        if value is None:
            value_str = "NA"
        else:
            value_str = str(value).strip() or "NA"

        if not field_name:
            field_name = kpi_desc

        records.append(
            {
                "kpi_description": kpi_desc,
                "field_name": field_name,
                "value": value_str,
            }
        )

    return records


In [10]:

# -----------------------------
# MAIN PIPELINE
# -----------------------------
gcs_client = get_gcs_client()
bucket = gcs_client.bucket(BUCKET_NAME)

blobs = list_ocr_json_blobs(bucket, RESULTS_PREFIX)
print(f"Found {len(blobs)} OCR JSON applicant files in gs://{BUCKET_NAME}/{RESULTS_PREFIX}")

all_records = []

for blob in blobs:
    print(f"Processing blob: {blob.name}")
    try:
        data = load_ocr_json(blob)
    except Exception as e:
        print(f"  -> Failed to parse JSON for {blob.name}: {e}")
        continue

    applicant_id = data.get("applicant_id") or infer_applicant_id(blob.name)
    documents = data.get("documents") or {}

    if not isinstance(documents, dict):
        print(f"  -> Skipping {blob.name}: 'documents' is not a dict")
        continue

    for doc_type, text in documents.items():
        document_label = str(doc_type)
        print(f"  Document: {document_label!r}, length of OCR text: {len(text or '')}")

        try:
            extracted_kpis = call_gemini_for_document(
                document_label=document_label,
                ocr_text=text or "",
                kpi_descriptions=all_kpis,
            )
        except Exception as e:
            print(f"    -> Gemini extraction failed for {document_label!r}: {e}")
            continue

        for entry in extracted_kpis:
            all_records.append(
                {
                    "applicant_id": applicant_id,
                    "document_type": document_label,
                    "kpi_description": entry["kpi_description"],
                    "field_name": entry["field_name"],
                    "value": entry["value"],
                }
            )

print(f"\nTotal extracted KPI records: {len(all_records)}")

results_df = pd.DataFrame(all_records)
display(results_df.head())

results_df.to_csv(OUTPUT_CSV_PATH, index=False)
print("Saved CSV to:", OUTPUT_CSV_PATH.resolve())


Found 4 OCR JSON applicant files in gs://capstone-ii-applicant-documents/vision-ocr-results/
Processing blob: vision-ocr-results/applicant_1/applicant_1.json
  Document: 'APPLICATION FORM_46', length of OCR text: 16778
  Document: 'DEVIATION APPROVAL MAILS_34', length of OCR text: 8534
  Document: 'DISBURSEMENT MEMO_43', length of OCR text: 2237
  Document: 'DISBURSEMENT MEMO_48', length of OCR text: 2241
  Document: 'DISBURSEMENT MEMO_6', length of OCR text: 2241
  Document: 'DISBURSEMENT REQUEST FORM_31', length of OCR text: 2072
  Document: 'DISBURSEMENT REQUEST FORM_47', length of OCR text: 3785
  Document: 'DRAFT SALE DEED - VETTED_63', length of OCR text: 4090
  Document: 'EXTERNAL FI REPORTS_14', length of OCR text: 429
  Document: 'EXTERNAL TECHNICAL VALUATION REPORT_18', length of OCR text: 4158
  Document: 'GECL-OFFER LETTER ACKNOWLEDGEMENT_15', length of OCR text: 19549
  Document: 'GECL-OFFER LETTER ACKNOWLEDGEMENT_56', length of OCR text: 19549
  Document: 'INSURANCE FORMS

,applicant_id,document_type,kpi_description,field_name,value
0,1,APPLICATION FORM_46,External FI reports /RCU Report,RCU Check Date,25 DEC 2023
1,1,APPLICATION FORM_46,Loan Application Form,Document Type,Loan Application Form
2,1,DEVIATION APPROVAL MAILS_34,Deviation Approval Mails,DeviationApprovalOutcome,Approved with 0.50% PF
3,1,DISBURSEMENT MEMO_43,Disbursement memo,Document Type,DISBURSEMENT MEMO
4,1,DISBURSEMENT MEMO_43,NACH and PDC,Payment Mode,NACH (E-NACH)


Saved CSV to: D:\FIU\Capstone II\capstone\data\gemini_llm_kpi_anymatch_extracted.csv
